# Batched molecular dynamics of the Li$_3$OCl teacher

`batched_md.py` advances B independent replicas with one model call per step; neighbour lists, integrator and bookkeeping reside on the GPU.

1. validation of energies and forces against the ASE calculator of the same model
2. throughput, integrator and neighbour lists included
3. a test run at 1000 K (32 replicas × 20 ps) with hop counting

Input: `step4_bundle.zip` with `batched_md.py` and `li3ocl_teacher.model` from this archive, uploaded when the notebook asks. Runtime: GPU. Results are checkpointed; running all cells again resumes an interrupted session.

In [ ]:
!pip -q install mace-torch ase
import torch, time, json, os, numpy as np
print(torch.__version__, torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU')

In [ ]:
if not os.path.exists('batched_md.py'):
    from google.colab import files
    up = files.upload()            # choose step4_bundle.zip
    os.system('unzip -o -q step4_bundle.zip')
print(sorted(os.listdir('.')))

In [ ]:
from ase import Atoms
from mace.calculators import MACECalculator
from batched_md import BatchedMD, HopCounter
DEV = 'cuda' if torch.cuda.is_available() else 'cpu'; A = 3.926
unit = Atoms('ClOLi3', scaled_positions=[(0,0,0), (.5,.5,.5), (.5,.5,0), (.5,0,.5), (0,.5,.5)], cell=[A]*3, pbc=True)
s = unit.repeat((3,3,3)); li_all = [i for i, z in enumerate(s.get_chemical_symbols()) if z == 'Li']; SITES = s.positions[li_all].copy(); del s[li_all[0]]
LI = np.array([i for i, z in enumerate(s.get_chemical_symbols()) if z == 'Li'])
FW = np.array([i for i, z in enumerate(s.get_chemical_symbols()) if z != 'Li'])
calc = MACECalculator(model_paths='li3ocl_teacher.model', device=DEV, default_dtype='float32'); model = calc.models[0]
rng = np.random.default_rng(0)
def start(B): return np.stack([s.positions + rng.normal(0, 0.05, s.positions.shape) for _ in range(B)])

In [ ]:
# (i) validation on this device
eng = BatchedMD(model, s.numbers, s.cell.lengths(), start(4), 1000.0, device=DEV, seed=1); eng.run(20); e, f = eng.energy_forces(); dE = dF = 0.0
for b in range(4):
    a = s.copy(); a.positions = eng.x[b].cpu().numpy(); a.wrap(); a.calc = calc
    dE = max(dE, abs(a.get_potential_energy() - float(e[b]))); dF = max(dF, float(np.abs(a.get_forces() - f[b].cpu().numpy()).max()))
print(f'validation: max |dE| = {dE:.2e} eV, max |dF| = {dF:.2e} eV/A'); VALID = dict(dE=dE, dF=dF)

In [ ]:
# (ii) real throughput, integrator and neighbour lists included
THR = []
for B in (8, 16, 32, 64):
    try:
        eng = BatchedMD(model, s.numbers, s.cell.lengths(), start(B), 1000.0, device=DEV, seed=B); eng.run(20)
        if DEV == 'cuda': torch.cuda.synchronize()
        t0 = time.time(); eng.run(100)
        if DEV == 'cuda': torch.cuda.synchronize()
        ms = (time.time() - t0) / 100 * 1e3
        THR.append(dict(B=B, ms_per_step=round(ms, 1), ms_per_replica_step=round(ms / B, 2), ns_per_day_total=round(86400 / ms * 1e3 * 2e-6 * B, 1))); print(THR[-1])
    except RuntimeError as err:
        print('B =', B, 'failed:', str(err)[:100]); break

In [ ]:
# (iii) pilot production at 1000 K with checkpoint / resume
T, B, NSTEPS, CHUNK = 1000.0, 32, 10000, 2500
ck = f'ckpt_{int(T)}K.pt'
eng = BatchedMD(model, s.numbers, s.cell.lengths(), start(B), T, device=DEV, seed=7); hc = HopCounter(eng, LI, SITES, framework_idx=FW, framework_ref=s.positions[FW].mean(0))
if os.path.exists(ck):
    st = torch.load(ck, weights_only=False); eng.x, eng.v, eng.step_count = st['x'].to(DEV), st['v'].to(DEV), st['step']; eng._build_neighbours(); eng.e, eng.f = eng.energy_forces()
    hc.assign, hc.hops, hc.x0, hc.msd, hc.events, hc.frames = st['assign'].to(DEV), st['hops'].to(DEV), st['x0'].to(DEV), st['msd'], st['events'], st['frames']; print('resumed at step', eng.step_count)
else:
    eng.run(1000); eng.step_count = 0; hc = HopCounter(eng, LI, SITES, framework_idx=FW, framework_ref=s.positions[FW].mean(0))             # equilibration, then reset the bookkeeping
t0 = time.time()
while eng.step_count < NSTEPS:
    eng.run(CHUNK, callback=hc, callback_every=10)
    torch.save(dict(x=eng.x.cpu(), v=eng.v.cpu(), step=eng.step_count, assign=hc.assign.cpu(), hops=hc.hops.cpu(), x0=hc.x0.cpu(), msd=hc.msd, events=hc.events, frames=hc.frames), ck)
    print(f'step {eng.step_count}: hops so far {int(hc.hops.sum())}, T = {eng.temperature().mean():.0f} K, mid-hop frames {len(hc.frames)}, {time.time()-t0:.0f} s', flush=True)

In [ ]:
ps = NSTEPS * 2e-3; msd = np.array(hc.msd); hops = hc.hops.cpu().numpy()
np.savez_compressed('li3ocl_pilot_1000K.npz', msd=msd, hops=hops, events=np.array(hc.events), frame_step=np.array([f[0] for f in hc.frames]), frame_replica=np.array([f[1] for f in hc.frames]),
                    frames=np.array([f[2] for f in hc.frames]) if hc.frames else np.zeros((0, len(s), 3), np.float32), sites=SITES, numbers=s.numbers, cell=s.cell.lengths())
summary = dict(gpu=torch.cuda.get_device_name(0) if DEV == 'cuda' else 'cpu', validation=VALID, throughput=THR, pilot=dict(T=T, replicas=B, ps_per_replica=ps,
               hops_total=int(hops.sum()), hops_per_ns=round(float(hops.sum() / (B * ps) * 1e3), 1), hops_per_replica=hops.tolist(), msd_end_A2=round(float(msd[-1].mean()), 3),
               mid_hop_frames=len(hc.frames), wall_s=round(time.time() - t0)))
print('=========== SUMMARY ==========='); print(json.dumps(summary, indent=1)); print('==============================')
os.system('zip -q li3ocl_step4_results.zip li3ocl_pilot_1000K.npz')
from google.colab import files; files.download('li3ocl_step4_results.zip')